# OTP specialization and win probability

This notebook estimates whether champion specialization is associated with winning after accounting for matchup strength, rank, lane, side, and champion. The goal is interpretation—not production match prediction.

The current patch sample is provisional. Rerun this notebook after expanding the Riot collection window.

## 1. Imports

If `statsmodels` is unavailable, run `%pip install statsmodels` once and restart the kernel.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')

def display_percentages(frame, columns, digits=2, signed=()):
    output = frame.copy()
    for column in columns:
        if column in output:
            sign = '+' if column in signed else ''
            output[column] = output[column].map(
                lambda value: f'{value:{sign}.{digits}%}' if pd.notna(value) else ''
            )
    display(output)

## 2. Load and prepare the modeling sample

The complete parquet remains unchanged. The 50-match matchup cutoff and all transformations happen in memory.

In [ ]:
start_dir = Path.cwd().resolve()
search_dirs = [start_dir, *start_dir.parents]
PROJECT_DIR = next(
    (path for path in search_dirs if (path / 'data' / 'processed').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find the repository data/processed directory.')

DATA_PATH = PROJECT_DIR / 'data' / 'processed' / 'otp_analysis_16_19.parquet'
raw_df = pd.read_parquet(DATA_PATH)

required = {
    'match_id', 'puuid', 'seed_tier', 'side', 'lane', 'champion', 'win',
    'champion_share', 'raw_matchup_winrate', 'matchup_games',
    'matchup_games_ge_50',
}
missing = required.difference(raw_df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

model_df = (
    raw_df.loc[raw_df['matchup_games_ge_50']].copy()
    .rename(columns={'seed_tier': 'rank', 'champion_share': 'otp_rate'})
)
model_df['win'] = model_df['win'].astype(int)
model_df['otp_rate'] = pd.to_numeric(model_df['otp_rate'], errors='coerce')
model_df['matchup_winrate'] = (
    pd.to_numeric(model_df['raw_matchup_winrate'], errors='coerce') / 100
)

# Scaling makes the coefficients readable.
# One unit of otp_rate_10pp = a 10-percentage-point increase in champion usage.
# One unit of matchup_winrate_1pp = a 1-percentage-point matchup advantage.
model_df['otp_rate_10pp'] = model_df['otp_rate'] * 10
model_df['matchup_winrate_1pp'] = model_df['matchup_winrate'] * 100

model_columns = [
    'match_id', 'puuid', 'win', 'otp_rate', 'otp_rate_10pp',
    'matchup_winrate', 'matchup_winrate_1pp',
    'rank', 'lane', 'side', 'champion',
]
model_df = model_df.loc[:, model_columns].dropna().reset_index(drop=True)

print('Source:', DATA_PATH)
print('Modeling rows:', f'{len(model_df):,}')
print('Unique matches:', f"{model_df['match_id'].nunique():,}")
print('Unique players:', f"{model_df['puuid'].nunique():,}")
display(model_df.head())

## 3. Modeling checks

The outcome must be binary, rates must be valid, and each sampled player should appear at most once per match.

In [ ]:
assert model_df['win'].isin([0, 1]).all()
assert model_df['otp_rate'].between(0, 1).all()
assert model_df['matchup_winrate'].between(0, 1).all()
assert model_df.duplicated(['match_id', 'puuid']).sum() == 0

audit = pd.Series({
    'player rows': len(model_df),
    'unique matches': model_df['match_id'].nunique(),
    'unique players': model_df['puuid'].nunique(),
    'actual win rate': model_df['win'].mean(),
    'average OTP rate': model_df['otp_rate'].mean(),
    'average raw matchup WR': model_df['matchup_winrate'].mean(),
})
display(audit.to_frame('value'))

## 4. Primary models: linear probability models

A linear probability model makes the OTP coefficient directly interpretable as a change in win probability. Standard errors are clustered by both match and player because multiple sampled observations can share a match and each player can appear repeatedly.

- **Model 0:** unadjusted OTP association
- **Model 1:** adds matchup, rank, lane, and side
- **Model 2:** additionally compares players while controlling for champion

In [ ]:
def two_way_cluster_codes(data, retained_index):
    # statsmodels' two-way cluster covariance requires numeric, aligned group IDs.
    retained = data.loc[retained_index, ['match_id', 'puuid']]
    if retained.isna().any().any():
        raise ValueError('Both cluster identifiers must be nonmissing.')
    if retained.nunique().min() < 2:
        raise ValueError('Two-way clustering needs at least two matches and players.')
    return np.column_stack([
        pd.factorize(retained[column], sort=False)[0]
        for column in ['match_id', 'puuid']
    ])


def fit_lpm(formula, data):
    model = smf.ols(formula, data=data)
    # Patsy may discard rows; align the cluster IDs with the observations it keeps.
    groups = two_way_cluster_codes(data, model.data.row_labels)
    return model.fit(
        cov_type='cluster',
        cov_kwds={'groups': groups, 'use_correction': True},
    )


formulas = {
    'Model 0: OTP only': 'win ~ otp_rate_10pp',
    'Model 1: core controls': (
        'win ~ otp_rate_10pp + matchup_winrate_1pp '
        '+ C(rank) + C(lane) + C(side)'
    ),
    'Model 2: + champion': (
        'win ~ otp_rate_10pp + matchup_winrate_1pp '
        '+ C(rank) + C(lane) + C(side) + C(champion)'
    ),
}

lpm_models = {name: fit_lpm(formula, model_df) for name, formula in formulas.items()}
print('Models fitted with match-and-player-clustered SE:', ', '.join(lpm_models))

In [ ]:
def otp_result_row(name, fitted_model):
    term = 'otp_rate_10pp'
    ci_low, ci_high = fitted_model.conf_int().loc[term]
    return {
        'model': name,
        'effect_pp_per_10pp_otp': fitted_model.params[term] * 100,
        'ci_low_pp': ci_low * 100,
        'ci_high_pp': ci_high * 100,
        'p_value': fitted_model.pvalues[term],
        'rows': int(fitted_model.nobs),
        'r_squared': fitted_model.rsquared,
    }

otp_model_results = pd.DataFrame([
    otp_result_row(name, fitted_model)
    for name, fitted_model in lpm_models.items()
])
display(otp_model_results)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_table = otp_model_results.iloc[::-1].reset_index(drop=True)
ax.errorbar(
    plot_table['effect_pp_per_10pp_otp'],
    plot_table['model'],
    xerr=[
        plot_table['effect_pp_per_10pp_otp'] - plot_table['ci_low_pp'],
        plot_table['ci_high_pp'] - plot_table['effect_pp_per_10pp_otp'],
    ],
    fmt='o',
    capsize=4,
)
ax.axvline(0, color='gray', linestyle='--')
ax.set(
    title='Estimated OTP-rate association with win probability',
    xlabel='Percentage-point change in win probability per +10pp OTP rate',
    ylabel='',
)
plt.tight_layout()
plt.show()

In [ ]:
main_lpm = lpm_models['Model 2: + champion']
effect = main_lpm.params['otp_rate_10pp'] * 100
ci_low, ci_high = main_lpm.conf_int().loc['otp_rate_10pp'] * 100
p_value = main_lpm.pvalues['otp_rate_10pp']
direction = 'increase' if effect >= 0 else 'decrease'

print(
    f'After adjustment, a 10-percentage-point increase in OTP rate is associated '
    f'with a {abs(effect):.2f}-percentage-point {direction} in win probability '
    f'(95% CI: {ci_low:.2f} to {ci_high:.2f}; p={p_value:.4f}).'
)

## 5. Logistic-regression robustness check

Logistic regression respects the 0–1 probability boundary. The code translates its result into the average predicted win-probability change from increasing OTP rate by 10 percentage points, holding other measured variables fixed. Coefficient standard errors use the same match-and-player clustering as the linear model. The linear model remains the primary estimate.

In [ ]:
logit_spec = smf.glm(
    formulas['Model 2: + champion'],
    data=model_df,
    family=sm.families.Binomial(),
)
logit_model = logit_spec.fit(
    cov_type='cluster',
    cov_kwds={
        'groups': two_way_cluster_codes(model_df, logit_spec.data.row_labels),
        'use_correction': True,
    },
)

# Compare each eligible row with itself at an OTP rate 10 percentage points higher.
eligible = model_df.loc[model_df['otp_rate'] <= 0.90].copy()
higher_otp = eligible.copy()
higher_otp['otp_rate'] += 0.10
higher_otp['otp_rate_10pp'] += 1
logit_effect_pp = (
    logit_model.predict(higher_otp) - logit_model.predict(eligible)
).mean() * 100

print(
    'Logistic-model average probability change for +10pp OTP rate: '
    f'{logit_effect_pp:+.2f} percentage points'
)
print(
    'Logistic OTP coefficient p-value (match-and-player-clustered): '
    f"{logit_model.pvalues['otp_rate_10pp']:.4f}"
)

## 6. Matchup equivalent of the OTP association

The estimated OTP coefficient can be expressed on the same scale as the matchup coefficient, holding champion and the other controls fixed. For a given main-versus-alternative OTP-rate gap, this is a **matchup equivalent**, not a complete champion-switching threshold. Switching champions also changes the champion indicator in Model 2; a champion-specific comparison would need that coefficient difference and credible data on both possible picks. This is an observational, population-level association.

Approximate 95% intervals use the delta method and the joint, two-way-clustered covariance of the OTP and matchup coefficients. A ratio is unstable if the matchup coefficient is not clearly positive, so the notebook withholds it in that case.

In [ ]:
otp_term = 'otp_rate_10pp'
matchup_term = 'matchup_winrate_1pp'
otp_coef = main_lpm.params[otp_term]
matchup_coef = main_lpm.params[matchup_term]
matchup_ci_low = main_lpm.conf_int().loc[matchup_term, 0]

if matchup_coef <= 0 or matchup_ci_low <= 0:
    matchup_ratio = np.nan
    matchup_ratio_se = np.nan
    print(
        'Matchup equivalent withheld: the adjusted matchup coefficient has '
        'not been estimated as clearly positive. Report both coefficients instead.'
    )
else:
    matchup_ratio = otp_coef / matchup_coef
    joint_cov = main_lpm.cov_params().loc[
        [otp_term, matchup_term], [otp_term, matchup_term]
    ].to_numpy()
    gradient = np.array([1 / matchup_coef, -otp_coef / matchup_coef**2])
    ratio_variance = float(gradient @ joint_cov @ gradient)
    if ratio_variance < -1e-12:
        raise ValueError('Two-way cluster covariance yielded a negative ratio variance.')
    matchup_ratio_se = np.sqrt(max(ratio_variance, 0.0))


def matchup_equivalent_interval(main_otp_rate, alternative_otp_rate):
    if not (0 <= alternative_otp_rate <= main_otp_rate <= 1):
        raise ValueError('Expected shares between 0 and 1, with main >= alternative.')
    if main_otp_rate + alternative_otp_rate > 1 + 1e-12:
        raise ValueError('Shares for one player cannot sum to more than 100%.')
    gap_in_10pp_units = (main_otp_rate - alternative_otp_rate) * 10
    if not np.isfinite(matchup_ratio):
        return pd.Series({'matchup_equivalent_pp': np.nan,
                          'ci_low_pp': np.nan, 'ci_high_pp': np.nan})
    estimate = gap_in_10pp_units * matchup_ratio
    margin = 1.96 * gap_in_10pp_units * matchup_ratio_se
    return pd.Series({'matchup_equivalent_pp': estimate,
                      'ci_low_pp': estimate - margin, 'ci_high_pp': estimate + margin})


example_pairs = pd.DataFrame({
    # Feasible shares for the same player: each pair sums to at most 100%.
    'main_otp_rate': [0.60, 0.70, 0.80, 0.90],
    'alternative_otp_rate': [0.40, 0.30, 0.20, 0.10],
})
example_pairs['otp_rate_gap'] = (
    example_pairs['main_otp_rate'] - example_pairs['alternative_otp_rate']
)
example_pairs = pd.concat([
    example_pairs,
    example_pairs.apply(
        lambda row: matchup_equivalent_interval(
            row['main_otp_rate'], row['alternative_otp_rate']
        ),
        axis=1,
    ),
], axis=1)
display_percentages(
    example_pairs,
    ['main_otp_rate', 'alternative_otp_rate', 'otp_rate_gap'],
    digits=0,
)
print('Matchup equivalent and approximate 95% interval are in percentage points.')

In [ ]:
otp_gaps = np.arange(0.10, 0.81, 0.10)
matchup_equivalent_curve = pd.DataFrame({
    'otp_rate_gap': otp_gaps,
    'matchup_equivalent_pp': [
        matchup_equivalent_interval(gap, 0)['matchup_equivalent_pp']
        for gap in otp_gaps
    ],
})

if matchup_equivalent_curve['matchup_equivalent_pp'].notna().any():
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.lineplot(
        data=matchup_equivalent_curve,
        x='otp_rate_gap',
        y='matchup_equivalent_pp',
        marker='o',
        ax=ax,
    )
    ax.xaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
    ax.set(
        title='Matchup equivalent of the estimated OTP association',
        xlabel='Main-versus-alternative OTP-rate gap',
        ylabel='Matchup equivalent (percentage points)',
    )
    plt.tight_layout()
    plt.show()

## 7. Does the OTP association differ by rank or lane?

Each interaction model starts from Model 2 and allows the OTP slope to vary by one group. A joint Wald test asks whether those slopes differ across the group's categories, using match-and-player-clustered uncertainty. Separate subgroup estimates below offer descriptive context; an interval crossing zero in one subgroup does not test whether two subgroups differ.

In [ ]:
def joint_otp_interaction_test(group):
    other_group = 'rank' if group == 'lane' else 'lane'
    formula = (
        f'win ~ otp_rate_10pp * C({group}) + matchup_winrate_1pp '
        f'+ C({other_group}) + C(side) + C(champion)'
    )
    fitted = fit_lpm(formula, model_df)
    interaction_terms = [
        term for term in fitted.params.index
        if term.startswith(f'otp_rate_10pp:C({group})[')
    ]
    if not interaction_terms:
        raise ValueError(f'No OTP-by-{group} interaction terms were estimated.')
    restriction = np.zeros((len(interaction_terms), len(fitted.params)))
    for i, term in enumerate(interaction_terms):
        restriction[i, fitted.params.index.get_loc(term)] = 1
    # Joint null: all nonreference OTP slopes match the reference group's slope.
    joint_test = fitted.wald_test(restriction, use_f=False, scalar=True)
    return {
        'group': group,
        'tested_interactions': len(interaction_terms),
        'wald_chi2': float(joint_test.statistic),
        'p_value': float(joint_test.pvalue),
        'rows': int(fitted.nobs),
    }


interaction_results = pd.DataFrame([
    joint_otp_interaction_test('lane'),
    joint_otp_interaction_test('rank'),
])
display(interaction_results)

In [ ]:
def subgroup_effects(data, group_column, group_values):
    rows = []
    for value in group_values:
        subset = data.loc[data[group_column] == value].copy()
        if len(subset) < 100:
            continue

        remaining_group = 'lane' if group_column == 'rank' else 'rank'
        formula = (
            'win ~ otp_rate_10pp + matchup_winrate_1pp '
            f'+ C({remaining_group}) + C(side) + C(champion)'
        )
        fitted = fit_lpm(formula, subset)
        ci_low, ci_high = fitted.conf_int().loc['otp_rate_10pp']
        rows.append({
            group_column: value,
            'player_rows': len(subset),
            'unique_players': subset['puuid'].nunique(),
            'effect_pp_per_10pp_otp': fitted.params['otp_rate_10pp'] * 100,
            'ci_low_pp': ci_low * 100,
            'ci_high_pp': ci_high * 100,
            'p_value': fitted.pvalues['otp_rate_10pp'],
        })
    return pd.DataFrame(rows)

rank_order = ['master', 'grandmaster', 'challenger']
lane_order = ['top', 'jungle', 'mid', 'adc', 'support']
rank_effects = subgroup_effects(model_df, 'rank', rank_order)
lane_effects = subgroup_effects(model_df, 'lane', lane_order)

print('By rank')
display(rank_effects)
print('By lane')
display(lane_effects)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for table, label, axis in [
    (rank_effects, 'rank', axes[0]),
    (lane_effects, 'lane', axes[1]),
]:
    axis.errorbar(
        table['effect_pp_per_10pp_otp'],
        table[label],
        xerr=[
            table['effect_pp_per_10pp_otp'] - table['ci_low_pp'],
            table['ci_high_pp'] - table['effect_pp_per_10pp_otp'],
        ],
        fmt='o',
        capsize=4,
    )
    axis.axvline(0, color='gray', linestyle='--')
    axis.set(
        title=f'OTP association by {label}',
        xlabel='Win-probability change (pp) per +10pp OTP rate',
        ylabel=label.title(),
    )
plt.tight_layout()
plt.show()

## Interpretation rules

1. Use **Model 2** as the main adjusted estimate and Model 0 only as the unadjusted comparison.
2. Interpret the OTP estimate as an **association**, not a causal effect. The models cluster inference by both match and player.
3. A confidence interval crossing zero means the current sample does not precisely distinguish the estimate from no association. Use the joint interaction tests for claims about differences across lanes or ranks.
4. The matchup equivalent holds champion fixed. Comparing two actual picks also requires their champion-coefficient contrast; the delta-method interval here only captures uncertainty in the OTP-to-matchup coefficient ratio.
5. `otp_rate` currently summarizes the short patch window and may use games occurring after a given match. A later version should construct pre-game specialization using only games available before each match.
6. Rerun the full collection and this notebook after more patch games accumulate before treating the estimates as final.